# Oral Lesion Screening from Smartphone Photos: Model v1
**Project:** Explainable multimodal AI for oral cancer and OPMD screening (M.Tech, IEEE paper)
**Notebook:** `02_model_v1_colab.ipynb` · Run in **Google Colab with a T4 GPU**

---

## Pipeline at a glance

| Step | What happens | Why |
|---|---|---|
| 1 | Setup: mount Drive, install libraries | Data lives in Google Drive |
| 2 | Download the **Karnataka** external test set | Unseen data from different hospitals and cameras |
| 3 | Index **SMART-OM + CODE** photos and assign labels | Training and validation data |
| 4 | Cache resized copies | Reading from Drive every epoch is too slow |
| 5 | **Patient-level** 5-fold split | The same person must never be in train and test |
| 6 | **Model A:** DINOv2 features + logistic regression | Strong, fast baseline for small data |
| 7 | **Model B:** fine-tuned EfficientNet-B0 | Main deep-learning model |
| 8 | **External test on Karnataka** | Shows whether the model generalises |
| 9 | **Grad-CAM++** heatmaps | Explainability: where the model looks |
| 10 | Results summary | One table for the paper |

## Datasets (all free access)
| Dataset | Role | Source |
|---|---|---|
| SMART-OM (Sci Data 2026) | train / validate | https://doi.org/10.6084/m9.figshare.31341790 |
| CODE v5 (Data in Brief 2026) | train / validate | https://doi.org/10.6084/m9.figshare.30550889 |
| Karnataka Oral Images (Mendeley, CC BY 4.0) | **external test only** | https://doi.org/10.17632/mhjyrn35p4 |

## Methods used
- DINOv2 (self-supervised vision foundation model): https://arxiv.org/abs/2304.07193
- EfficientNet: https://arxiv.org/abs/1905.11946
- Grad-CAM++: https://arxiv.org/abs/1710.11063
- StratifiedGroupKFold (patient-level splitting): https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. The first run takes about 30–45 min; re-runs reuse the caches saved in Drive.

---
## Step 1 · Setup
**What:** connects Colab to your Google Drive and installs two libraries: `timm` (pretrained image models) and `grad-cam` (heatmaps).

**Output to expect:** a Drive permission pop-up (click *Allow*), then a list of folders in `data/raw`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install timm grad-cam

In [ ]:
import os, re, glob, json, zipfile, shutil, random, hashlib, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = '/content/drive/MyDrive/Oral_cancer_ai'
RAW, PROC = f'{ROOT}/data/raw', f'{ROOT}/data/processed'
RES, MODELS = f'{ROOT}/results', f'{ROOT}/models'
for d in [PROC, f'{RES}/tables', f'{RES}/figures', MODELS, f'{RAW}/karnataka_mendeley']:
    os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
N_FOLDS = 5
print('raw folders:', os.listdir(RAW))

---
## Step 2 · Download the Karnataka external test set
**What:** downloads only the `original_data` folder (165 benign, 158 malignant photos) from Mendeley into
`Drive/Oral_cancer_ai/data/raw/karnataka_mendeley/`. The `augmented_data` folder is **skipped**, because augmented copies of
the same photos would make the test look easier than it is.

**Why it's external:** different hospitals (Karnataka), different cameras (mobile + intraoral), different team.
It is **never used for training or for choosing thresholds**, only for the final test in Step 8.

**Important difference:** Karnataka labels are *benign lesion vs malignant*. It has no normal/OPMD classes, so it
tests the **binary** model only, and its negatives are benign lesions (harder than normal mouths).

**If the automatic download fails:** open https://data.mendeley.com/datasets/mhjyrn35p4/2, click *Download All*,
and upload the zip into `Oral_cancer_ai/data/raw/karnataka_mendeley/` in Drive. The cell unzips it on the next run.

In [ ]:
import requests
KDIR = f'{RAW}/karnataka_mendeley'
API = 'https://data.mendeley.com/public-api/datasets/mhjyrn35p4'
VERSION = 2

def have_images(d):
    return any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for _, _, fs in os.walk(d) for f in fs)

# 1) a manually uploaded zip? unzip it
for z in glob.glob(f'{KDIR}/*.zip'):
    if not have_images(KDIR):
        print('unzipping', os.path.basename(z)); zipfile.ZipFile(z).extractall(KDIR)

# 2) otherwise try the Mendeley public API (original_data only)
if not have_images(KDIR):
    try:
        folders = requests.get(f'{API}/folders/{VERSION}', timeout=60).json()
        byid = {f['id']: f for f in folders}
        def fullpath(f):
            parts = [f['name']]
            while f.get('parent_id') in byid: f = byid[f['parent_id']]; parts.append(f['name'])
            return '/'.join(reversed(parts))
        targets = [(f['id'], fullpath(f)) for f in folders if 'original' in fullpath(f).lower()]
        n = 0
        for fid, fpath in targets:
            files = requests.get(f'{API}/files', params={'folder_id': fid, 'version': VERSION}, timeout=60).json()
            for fl in files:
                url = fl.get('content_details', {}).get('download_url')
                if not url: continue
                out = os.path.join(KDIR, fpath, fl['filename']); os.makedirs(os.path.dirname(out), exist_ok=True)
                if not os.path.exists(out):
                    open(out, 'wb').write(requests.get(url, timeout=120).content); n += 1
        print('downloaded', n, 'files')
    except Exception as e:
        print('⚠ automatic download failed:', e, '\n→ use the manual option described above, then re-run this cell')

kar = [os.path.join(dp, f) for dp, _, fs in os.walk(KDIR) for f in fs
       if f.lower().endswith(('.jpg', '.jpeg', '.png')) and 'augment' not in dp.lower() and '__MACOSX' not in dp]
print('Karnataka original images found:', len(kar), '(expected 323)')

---
## Step 3 · Index SMART-OM + CODE and assign labels
**What:** walks through both datasets, keeps only the **unannotated** photos (the annotated copies have drawings on them), and reads the
class from the folder name and the patient ID (`SMITA00123`) from the file name.

**Labels used:**
| Original class | Binary task | 3-class triage |
|---|---|---|
| Normal | 0 · not suspicious | Normal |
| Variation from normal (Fordyce spots, leukoedema…) | 0 · not suspicious | Variation |
| OPMD (leukoplakia, OSMF, lichen planus…) | **1 · suspicious** | Suspicious |
| Oral cancer | **1 · suspicious** | Suspicious |

**Output to check:** the tables of images and patients per class. Expected roughly: SMART-OM 2,145 / 179 / 125 / 20 and
CODE 5,424 / 166 / 14 / 7 (Normal / Variation / OPMD / Cancer). If any folder is reported as *class not recognised*, tell me.

In [ ]:
IMG = ('.jpg', '.jpeg', '.png')

def get_class(path):
    s = path.lower()
    if 'cancer' in s or 'oscc' in s or 'malignan' in s: return 'Cancer'
    if 'opmd' in s or 'potentially' in s:               return 'OPMD'
    if 'variation' in s or 'variant' in s:              return 'Variation'   # must be checked before 'normal'
    if 'normal' in s:                                   return 'Normal'
    return None

def get_pid(fn):
    m = re.search(r'SMITA[_\- ]?0*(\d+)', fn, re.I)
    return f'SMITA{int(m.group(1)):05d}' if m else None

def get_site(folder):
    return re.sub(r'^\s*\d+\.\s*', '', os.path.basename(folder)).strip()

rows = []
for ds, sub in [('SMART-OM', 'smart_om'), ('CODE', 'code')]:
    base = f'{RAW}/{sub}'
    for dp, _, fs in os.walk(base):
        if '__MACOSX' in dp: continue
        rel = os.path.relpath(dp, base)
        if 'unann' not in re.sub(r'[\s\-_]', '', rel.lower()): continue
        for fn in fs:
            if fn.lower().endswith(IMG) and not fn.startswith('._'):
                rows.append(dict(dataset=ds, path=os.path.join(dp, fn), file=fn, folder=rel,
                                 cls=get_class(rel), site=get_site(rel), patient_id=get_pid(fn)))
df = pd.DataFrame(rows)
print('images found:', len(df))

unk = df[df.cls.isna()].folder.value_counts()
if len(unk): print('\n⚠ class not recognised for these folders (dropped):\n', unk.head(20))
df = df[df.cls.notna()].copy()

# images without a SMITA ID get a conservative fallback group (keeps look-alike files together)
miss = df.patient_id.isna()
df.loc[miss, 'patient_id'] = (df.loc[miss, 'dataset'] + '_' + df.loc[miss, 'cls'] + '_' +
                              df.loc[miss, 'file'].str.split(r'[_\-\s.]').str[0])
print('images without SMITA ID (fallback group):', int(miss.sum()))

df = df.drop_duplicates('path').reset_index(drop=True)
df['y_bin'] = df.cls.isin(['OPMD', 'Cancer']).astype(int)
df['y3']    = df.cls.map({'Normal': 0, 'Variation': 1, 'OPMD': 2, 'Cancer': 2})
CLASS3 = ['Normal', 'Variation', 'Suspicious']

print('\nImages per class:'); print(pd.crosstab(df.cls, df.dataset, margins=True))
print('\nPatients per class:'); print(df.groupby(['dataset', 'cls']).patient_id.nunique().unstack(0))

---
## Step 4 · Cache resized copies
**What:** corrects photo rotation (phone EXIF tag), shrinks each photo to at most 448 px and saves it locally, plus a zip backup
in Drive (`data/processed/img448_cache.zip`).

**Why:** reading ~8,000 full-size photos from Drive every epoch would take hours. With the cache, a training epoch takes seconds.

In [ ]:
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor

CACHE, CZIP = '/content/img448', f'{PROC}/img448_cache.zip'
os.makedirs(CACHE, exist_ok=True)
if os.path.exists(CZIP) and not os.listdir(CACHE):
    print('restoring cache from Drive…'); zipfile.ZipFile(CZIP).extractall(CACHE)

cache_name = lambda p: f"{CACHE}/{hashlib.md5(p.encode()).hexdigest()[:16]}.jpg"
def prep(a):
    src, dst = a
    if os.path.exists(dst): return True
    try:
        im = ImageOps.exif_transpose(Image.open(src)).convert('RGB')
        im.thumbnail((448, 448)); im.save(dst, quality=92); return True
    except Exception as e:
        print('bad image:', src, e); return False

df['cache'] = [cache_name(p) for p in df.path]
kdf = pd.DataFrame({'path': kar})
kdf['cls'] = np.where(kdf.path.str.lower().str.contains('malignan'), 'Malignant',
             np.where(kdf.path.str.lower().str.contains('benign'), 'Benign', None))
kdf = kdf[kdf.cls.notna()].copy()
kdf['y_bin'] = (kdf.cls == 'Malignant').astype(int)
kdf['cache'] = [cache_name(p) for p in kdf.path]

t = time.time()
with ThreadPoolExecutor(16) as ex:
    ok  = list(ex.map(prep, zip(df.path, df.cache)))
    okk = list(ex.map(prep, zip(kdf.path, kdf.cache)))
df, kdf = df[ok].reset_index(drop=True), kdf[okk].reset_index(drop=True)
print(f'cached {len(df)} SMART-OM/CODE + {len(kdf)} Karnataka images in {time.time()-t:.0f}s')

# remove exact duplicate photos inside Karnataka (same bytes)
kdf['md5'] = [hashlib.md5(open(p, 'rb').read()).hexdigest() for p in kdf.path]
before = len(kdf); kdf = kdf.drop_duplicates('md5').reset_index(drop=True)
print('Karnataka:', kdf.cls.value_counts().to_dict(), f'| exact duplicates removed: {before-len(kdf)}')

if not os.path.exists(CZIP):
    shutil.make_archive(CZIP[:-4], 'zip', CACHE); print('cache saved to Drive')

---
## Step 5 · Patient-level 5-fold split
**What:** divides **patients** (not photos) into 5 groups, keeping the share of suspicious cases similar in each.
Each fold takes a turn as the test set.

**Why it matters:** one patient has up to 8 photos (lips, cheeks, tongue, arches). If photos of the same person sat in both
train and test, the model could recognise the *person* instead of the *lesion*, and the scores would be falsely high.
The `assert` line stops the notebook if any patient leaks across folds.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df['fold'] = -1
for k, (_, te) in enumerate(sgkf.split(df, df.y_bin, groups=df.patient_id)):
    df.loc[te, 'fold'] = k
assert df.groupby('patient_id').fold.nunique().max() == 1, 'patient leakage!'
print(pd.crosstab(df.fold, df.cls))
df.drop(columns=['path']).to_csv(f'{PROC}/model_v1_index_folds.csv', index=False)

---
## Step 6 · Model A: DINOv2 features + logistic regression
**What:** a large pretrained vision model (DINOv2 ViT-B/14) turns each photo into a 768-number "description" (embedding).
A simple logistic regression then learns to separate suspicious from not suspicious using those numbers.

**Why start here:** with only ~170 suspicious photos, training a deep network from scratch overfits. A frozen foundation model
plus a simple classifier is robust, fast, and a fair baseline that reviewers expect.

**Class imbalance:** about 1 suspicious photo per 40, so `class_weight='balanced'` makes each suspicious photo count more.

**Threshold:** for screening, missing a case is worse than a false alarm, so the decision threshold is set for
**sensitivity ≥ 90%**. It is chosen *inside the training folds only* (inner cross-validation), never on the test fold.

**How to read the output:**
- **AUC:** overall ranking ability (0.5 = coin flip, 1.0 = perfect)
- **AUPRC:** better than AUC when positives are rare; compare it with the positive rate (≈0.02–0.03 means no skill)
- **Flag rate by true class:** how often each class is flagged as suspicious. A high flag rate for *Variation* means look-alike false alarms.

In [ ]:
import torch, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '| if this says cpu, switch the runtime to T4 GPU')
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), NORM])

class ImgDS(Dataset):
    def __init__(s, paths, ys=None, tf=eval_tf): s.p, s.y, s.tf = list(paths), ys, tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        x = s.tf(Image.open(s.p[i]).convert('RGB'))
        return (x, torch.tensor(float(s.y[i]))) if s.y is not None else x

def dino_features(paths, cache_file):
    if os.path.exists(cache_file) and list(np.load(cache_file, allow_pickle=True)['cache']) == list(paths):
        return np.load(cache_file, allow_pickle=True)['X']
    global dino
    if 'dino' not in globals():
        dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').to(dev).eval()
    feats = []
    with torch.no_grad(), torch.autocast(dev, enabled=(dev == 'cuda')):
        for xb in DataLoader(ImgDS(paths), batch_size=64, num_workers=2):
            feats.append(dino(xb.to(dev)).float().cpu().numpy())
    X_ = np.concatenate(feats); np.savez(cache_file, X=X_, cache=np.array(list(paths))); return X_

X  = dino_features(df.cache,  f'{PROC}/dinov2_vitb14_emb.npz')
XK = dino_features(kdf.cache, f'{PROC}/dinov2_vitb14_emb_karnataka.npz')
print('features:', X.shape, XK.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             confusion_matrix, f1_score, ConfusionMatrixDisplay)

TARGET_SENS = 0.90
def thr_for_sens(y, p, target=TARGET_SENS):
    fpr, tpr, thr = roc_curve(y, p)
    return thr[np.argmax(tpr >= target)]

def metrics(y, p, thr):
    pr = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0, 1]).ravel()
    return dict(n=len(y), pos=int(y.sum()), AUC=roc_auc_score(y, p), AUPRC=average_precision_score(y, p),
                sensitivity=tp / (tp + fn), specificity=tn / (tn + fp))

def make_lr(): return make_pipeline(StandardScaler(),
                                    LogisticRegression(C=0.05, class_weight='balanced', max_iter=5000))

y, g = df.y_bin.values, df.patient_id.values
oof, pred, rows = np.zeros(len(df)), np.zeros(len(df), int), []
for k in range(N_FOLDS):
    tr, te = (df.fold != k).values, (df.fold == k).values
    inner = cross_val_predict(make_lr(), X[tr], y[tr], groups=g[tr], cv=GroupKFold(3), method='predict_proba')[:, 1]
    thr = thr_for_sens(y[tr], inner)
    oof[te] = make_lr().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    pred[te] = (oof[te] >= thr).astype(int)
    rows.append(dict(fold=k, **metrics(y[te], oof[te], thr)))
resA = pd.DataFrame(rows)
print(resA.round(3)); print('\nmean ± sd'); print(resA[['AUC', 'AUPRC', 'sensitivity', 'specificity']].agg(['mean', 'std']).round(3))

df['A_prob'], df['A_pred'] = oof, pred
print('\nFlag rate by true class (Model A):'); print(df.groupby('cls').A_pred.mean().round(3))
print('\nAUC per dataset:')
for ds_, d in df.groupby('dataset'):
    if d.y_bin.nunique() == 2: print(f'  {ds_}: {roc_auc_score(d.y_bin, d.A_prob):.3f}  (pos={d.y_bin.sum()})')
resA.to_csv(f'{RES}/tables/modelA_dinov2_binary_folds.csv', index=False)

### Step 6b · 3-class triage: Normal / Variation / Suspicious
**Why:** in real screening, harmless variations get mistaken for pre-cancer and overload referral clinics. Telling them apart
is clinically useful, and few datasets label variations separately (SMART-OM and CODE do).

**How to read:** the confusion matrix is row-normalised. Each row is a true class, and the diagonal is the share predicted correctly.
**Macro-F1** averages performance across the three classes equally, so the big Normal class can't hide poor results elsewhere.

In [ ]:
y3, oof3 = df.y3.values, np.zeros((len(df), 3))
for k in range(N_FOLDS):
    tr, te = (df.fold != k).values, (df.fold == k).values
    oof3[te] = make_lr().fit(X[tr], y3[tr]).predict_proba(X[te])
p3 = oof3.argmax(1)
print('macro-F1:', round(f1_score(y3, p3, average='macro'), 3))
print('macro AUC (one-vs-rest):', round(roc_auc_score(y3, oof3, multi_class='ovr', average='macro'), 3))
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix(y3, p3, normalize='true'), display_labels=CLASS3).plot(
    ax=ax, values_format='.2f', cmap='Blues', colorbar=False)
ax.set_title('Model A · 3-class (row-normalised)'); plt.tight_layout()
plt.savefig(f'{RES}/figures/modelA_3class_confusion.png', dpi=200); plt.show()

---
## Step 7 · Model B: fine-tuned EfficientNet-B0
**What:** an ImageNet-pretrained EfficientNet-B0 is fine-tuned end-to-end. Split: **fold 0 = test, fold 1 = validation,
folds 2–4 = training** (still patient-level).

**Imbalance handling:** a *weighted sampler* draws suspicious photos so they make up about 25% of every training batch, and
augmentation (random crop, flip, rotation, colour jitter) means repeated suspicious photos look slightly different each time.

**Model selection:** the epoch with the best **validation AUPRC** is saved to `Drive/Oral_cancer_ai/models/effb0_binary_v1.pt`.

**How to read the log:** loss should go down; validation AUPRC should rise, then flatten. If it falls while loss keeps
dropping, that's overfitting, and the saved checkpoint is the best epoch before it.

In [ ]:
import timm, torch.nn as nn
from torch.utils.data import WeightedRandomSampler

TEST_FOLD, VAL_FOLD = 0, 1
EPOCHS, BS, LR, POS_SHARE = 15, 32, 3e-4, 0.25

train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
                      T.RandomHorizontalFlip(), T.RandomRotation(15),
                      T.ColorJitter(0.2, 0.2, 0.15, 0.03), T.ToTensor(), NORM])
trd = df[~df.fold.isin([TEST_FOLD, VAL_FOLD])]; vad = df[df.fold == VAL_FOLD]; ted = df[df.fold == TEST_FOLD].copy()
print(f'train {len(trd)} (pos {trd.y_bin.sum()}) | val {len(vad)} (pos {vad.y_bin.sum()}) | test {len(ted)} (pos {ted.y_bin.sum()})')

npos = trd.y_bin.sum(); nneg = len(trd) - npos
w = np.where(trd.y_bin == 1, POS_SHARE / npos, (1 - POS_SHARE) / nneg)
sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(trd), replacement=True)
dl_tr = DataLoader(ImgDS(trd.cache, trd.y_bin.values, train_tf), batch_size=BS, sampler=sampler, num_workers=2, drop_last=True)
dl_va = DataLoader(ImgDS(vad.cache, vad.y_bin.values), batch_size=64, num_workers=2)
dl_te = DataLoader(ImgDS(ted.cache, ted.y_bin.values), batch_size=64, num_workers=2)
dl_k  = DataLoader(ImgDS(kdf.cache, kdf.y_bin.values), batch_size=64, num_workers=2)

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=1).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl_tr))
scaler = torch.cuda.amp.GradScaler(enabled=(dev == 'cuda'))
lossf = nn.BCEWithLogitsLoss()

def predict(dl):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(dev, enabled=(dev == 'cuda')):
        for xb, _ in dl: out.append(torch.sigmoid(model(xb.to(dev)).float()).squeeze(1).cpu().numpy())
    return np.concatenate(out)

best, BEST_PATH, hist = -1, f'{MODELS}/effb0_binary_v1.pt', []
for ep in range(EPOCHS):
    model.train(); tl = 0
    for xb, yb in dl_tr:
        xb, yb = xb.to(dev), yb.to(dev)
        with torch.autocast(dev, enabled=(dev == 'cuda')):
            loss = lossf(model(xb).squeeze(1), yb)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item()
    pv = predict(dl_va)
    auc, auprc = roc_auc_score(vad.y_bin, pv), average_precision_score(vad.y_bin, pv)
    hist.append((ep, tl / len(dl_tr), auc, auprc)); flag = ''
    if auprc > best: best = auprc; torch.save(model.state_dict(), BEST_PATH); flag = '  ← saved'
    print(f'epoch {ep+1:2d}  loss {tl/len(dl_tr):.3f}  val AUC {auc:.3f}  val AUPRC {auprc:.3f}{flag}')

### Step 7b · Internal test (fold 0)
The threshold comes from the **validation** fold (sensitivity ≥ 90%), then is applied unchanged to the test fold.
Model A is scored on the same fold so the comparison is fair.

In [ ]:
model.load_state_dict(torch.load(BEST_PATH))
pv, pt = predict(dl_va), predict(dl_te)
thrB = thr_for_sens(vad.y_bin.values, pv)
ted['B_prob'], ted['B_pred'] = pt, (pt >= thrB).astype(int)
ta = (df.fold == TEST_FOLD).values
internal = pd.DataFrame([
    dict(model='A · DINOv2+LR', test='internal fold 0', **metrics(df.y_bin.values[ta], df.A_prob.values[ta],
                                                                   thr_for_sens(df.y_bin.values[~ta], df.A_prob.values[~ta]))),
    dict(model='B · EfficientNet-B0', test='internal fold 0', **metrics(ted.y_bin.values, pt, thrB))])
print(internal.round(3).to_string(index=False))
print('\nFlag rate by true class (Model B):'); print(ted.groupby('cls').B_pred.mean().round(3))

---
## Step 8 · External test on Karnataka (unseen hospitals and cameras)
**What:** both models are applied **unchanged** to the Karnataka photos.
- Model A is refit on *all* SMART-OM + CODE data; its threshold comes from the out-of-fold predictions in Step 6 (training data only).
- Model B uses the saved checkpoint and its validation threshold from Step 7.

**How to read:** a drop from internal to external performance is normal and **must be reported honestly**; it shows how much
the model depends on the training hospital's cameras and patients. Remember that Karnataka negatives are *benign lesions*,
not normal mouths, which makes this a harder test.

In [ ]:
clfA_all = make_lr().fit(X, y)
thrA_all = thr_for_sens(y, df.A_prob.values)
pkA = clfA_all.predict_proba(XK)[:, 1]
pkB = predict(dl_k)
kdf['A_prob'], kdf['B_prob'] = pkA, pkB
external = pd.DataFrame([
    dict(model='A · DINOv2+LR', test='external Karnataka', **metrics(kdf.y_bin.values, pkA, thrA_all)),
    dict(model='B · EfficientNet-B0', test='external Karnataka', **metrics(kdf.y_bin.values, pkB, thrB))])
print(external.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for name, yy, pp in [('A internal', df.y_bin.values[ta], df.A_prob.values[ta]), ('B internal', ted.y_bin.values, pt),
                     ('A external', kdf.y_bin.values, pkA), ('B external', kdf.y_bin.values, pkB)]:
    f_, t_, _ = roc_curve(yy, pp)
    ax[0 if 'internal' in name else 1].plot(f_, t_, label=f'{name} (AUC {roc_auc_score(yy, pp):.2f})')
for a, tt in zip(ax[:2], ['ROC · internal test (fold 0)', 'ROC · external (Karnataka)']):
    a.plot([0, 1], [0, 1], ':', c='grey'); a.set(xlabel='1 − specificity', ylabel='sensitivity', title=tt); a.legend(fontsize=8)
h = np.array(hist); ax[2].plot(h[:, 0] + 1, h[:, 3], marker='o'); ax[2].set(xlabel='epoch', ylabel='val AUPRC', title='Model B training')
plt.tight_layout(); plt.savefig(f'{RES}/figures/model_v1_roc_internal_external.png', dpi=200); plt.show()

---
## Step 9 · Grad-CAM++: where does the model look?
**What:** Grad-CAM++ colours the parts of the photo that pushed Model B towards "suspicious" (red = strong influence).

**Shown:** top row = original photo, bottom row = heatmap, for
(1) suspicious internal cases the model caught, (2) harmless variations it worried about (look-alikes), and
(3) Karnataka malignant cases.

**What a good result looks like:** heat on the lesion itself, not on teeth, lips, fingers, cheek retractors or the photo edge.
Heat on background objects is a **shortcut** and should be discussed in the paper. Later, these maps will be compared
numerically with the doctors' lesion outlines (SMART-OM / CODE annotations).

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

cam = GradCAMPlusPlus(model=model.eval(), target_layers=[model.blocks[-1]])
show = pd.concat([ted[ted.y_bin == 1].nlargest(4, 'B_prob').assign(src='internal'),
                  ted[(ted.y_bin == 0) & (ted.cls == 'Variation')].nlargest(2, 'B_prob').assign(src='internal'),
                  kdf[kdf.y_bin == 1].nlargest(3, 'B_prob').assign(src='Karnataka')])
fig, axes = plt.subplots(2, len(show), figsize=(2.3 * len(show), 4.9))
for j, (_, r) in enumerate(show.iterrows()):
    im = Image.open(r.cache).convert('RGB').resize((224, 224)); rgb = (np.asarray(im) / 255.0).astype(np.float32)
    heat = cam(input_tensor=eval_tf(im).unsqueeze(0).to(dev), targets=[ClassifierOutputTarget(0)])[0]
    axes[0, j].imshow(rgb); axes[1, j].imshow(show_cam_on_image(rgb, heat, use_rgb=True))
    axes[0, j].set_title(f'{r.src}\n{r.cls} · p={r.B_prob:.2f}', fontsize=8)
    for a in axes[:, j]: a.axis('off')
plt.tight_layout(); plt.savefig(f'{RES}/figures/model_v1_gradcam.png', dpi=200); plt.show()

---
## Step 10 · Results summary
One table with internal and external results for both models, saved to
`Drive/Oral_cancer_ai/results/tables/model_v1_summary.csv` (ready for the paper or slides).

In [ ]:
cv = resA[['AUC', 'AUPRC', 'sensitivity', 'specificity']].mean()
summary = pd.concat([
    pd.DataFrame([dict(model='A · DINOv2+LR', test='internal 5-fold CV (mean)', n=len(df), pos=int(y.sum()), **cv)]),
    internal, external], ignore_index=True)
summary.to_csv(f'{RES}/tables/model_v1_summary.csv', index=False)
print(summary.round(3).to_string(index=False))

---
## Notes for interpretation and next steps
- **Positives are rare** (OPMD + cancer ≈ 2–3% of SMART-OM + CODE photos), so read **AUPRC and sensitivity at the fixed threshold**, not accuracy.
- **No threshold was ever tuned on a test set** (internal or external).
- **Limitations to state:** few cancer photos (~27); both training datasets come from one ICMR SMITA programme (Tamil Nadu);
  Karnataka has no patient IDs and only benign-vs-malignant labels.
- **Next notebooks:** `03` image + habit fusion (with SHAP for habits) · `04` habit-from-photo on CODE normal-looking mouths
  (main novelty) · `05` Grad-CAM vs expert lesion masks (quantitative explainability).